# SPORE quickstart

This notebook walks through the core workflow:

1. Loading a detector IRF and inspecting the effective area
2. Point source simulation (steady-state and atmospheric background)
3. Extended source simulation (isotropic astrophysical)
4. Sanity checks: Poisson statistics, Eddington bias, angular resolution vs energy
5. Multi-detector joint simulation
6. Saving and loading events

Section 3 also covers good run lists (§3.4) — sampling with realistic detector uptime gaps.

All paths are resolved relative to the repository root, so the notebook can be run from any working directory.

## 1  Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import h5py
import tempfile
from pathlib import Path

import spore
from spore import (
    Detector, PointSource, ExtendedSource,
    SourceSampler, GoodRunList, SkyCoordinate, ureg,
)

# Locate the repository root (works regardless of working directory)
_p = Path.cwd()
while not (_p / 'pyproject.toml').exists() and _p != _p.parent:
    _p = _p.parent
REPO_ROOT = _p
RESPONSE_FILE = str(REPO_ROOT / 'resources' / 'configs' / 'ps10yr_detector_response.h5')
print('repo root   :', REPO_ROOT)
print('response exists:', Path(RESPONSE_FILE).exists())

## 2  Loading a detector

`Detector.from_config` accepts a plain dictionary.  We use the IceCube 10-year point-source sample response built by `scripts/build_ps10yr_detector_response.py`.

In [ ]:
south_polar = Detector.from_config({
    'properties': {'latitude': -90.0, 'longitude': 0.0, 'medium': 'Ice'},
    'response':   {'detector_response_file': RESPONSE_FILE},
})

print('Location  : lat={:.1f} deg, lon={:.1f} deg'.format(
    np.degrees(south_polar.location.latitude), np.degrees(south_polar.location.longitude)))
print('Morphologies:', south_polar.response.available_morphologies)
effa = south_polar.response.effective_area['track']
print('A_eff range : {:.0f} GeV -- {:.2e} GeV'.format(
    effa.e_min_gev, effa.e_max_gev))

### 2.1  Effective area vs energy

In [ ]:
effa = south_polar.response.effective_area['track']
energies = np.logspace(np.log10(effa.e_min_gev), 7, 200)

zenith_cases = [
    (np.pi * 0.5, 'zen=90 deg (horizontal)'),
    (np.pi * 0.6, 'zen=108 deg'),
    (np.pi * 0.7, 'zen=126 deg'),
    (np.pi * 0.8, 'zen=144 deg'),
    (np.pi * 0.9, 'zen=162 deg'),
    (np.pi, 'zen=180 deg (upgoing)'),
]

fig, ax = plt.subplots(figsize=(8, 4))
for zen, label in zenith_cases:
    vals = effa(np.full(len(energies), zen), energies)
    ax.step(energies / 1e3, vals, label=label, where="mid")

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Energy [TeV]')
ax.set_ylabel('A_eff [cm^2]')
ax.set_title('Track effective area -- IceCube PS-10yr')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 2.2  PSF median angular resolution vs energy

When the detector response stores a standalone `angular_response` callable, `angular_response(E, 0.5)` gives the median PSF deflection at energy E.  The IceCube PS-10yr response uses a joint smearing table instead, so no standalone angular-response callable is available here; we skip this plot and demonstrate angular resolution empirically in section 5.2.

In [ ]:
ang = south_polar.response.angular_response['track']
if ang is not None:
    e_grid = np.logspace(np.log10(effa.e_min_gev), np.log10(effa.e_max_gev), 80)
    median_psf_deg = np.degrees([ang(e, 0.5) for e in e_grid])

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(e_grid / 1e3, median_psf_deg, color='steelblue')
    ax.set_xscale('log')
    ax.set_xlabel('Energy [TeV]')
    ax.set_ylabel('Median PSF [deg]')
    ax.set_title('Track angular resolution (median) -- IceCube PS-10yr')
    plt.tight_layout()
    plt.show()
else:
    print('angular_response not available for this detector response '
          '(joint smearing IRF); see section 5.2 for empirical PSF.')

## 3  Point source simulation

We simulate TXS 0506+056 (dec +5.7 deg), the first IceCube blazar candidate, with a soft E^-2.7 power-law spectrum.  `n_time_samples=100` enables steady-state mode: the effective area is averaged over 100 hour-angle samples uniformly spaced over one diurnal cycle.  This is the correct treatment for observations spanning many sidereal days.

In [ ]:
src = PointSource.from_config({
    'flux': {
        'norm':  2.5e-14,   # GeV^-1 cm^-2 s^-1 per species
        'gamma': 3.2,
        'pivot': 1e3,     # 1 TeV
        'emin':  1e2,
        'emax':  1e7,
    },
    'location': {'declination': -0.013, 'right_ascension': 77.4},
})
print('Source: dec={:.2f} deg, RA={:.2f} deg'.format(
    np.degrees(src.location.declination),
    np.degrees(src.location.right_ascension)))

In [ ]:
sampler_ps = SourceSampler(south_polar, src, n_time_samples=100)

one_year = ureg.Quantity(365.25, 'day')
n_exp = sampler_ps.expected_events('track', one_year)
print(f'Expected track events in one year: {n_exp:.2f}')

### 3.1  Sample ten years of events (Poisson mode)

In [ ]:
events_ps = sampler_ps.sample_events('track', deltat=10 * one_year, seed=1)
print(f'Sampled {len(events_ps)} events')

true_e_tev  = np.array([ev.true_energy.to('TeV').magnitude  for ev in events_ps])
reco_e_tev  = np.array([ev.reco_energy.to('TeV').magnitude  for ev in events_ps])
reco_decs   = np.degrees([ev.reco_direction.declination      for ev in events_ps])
reco_ras    = np.degrees([ev.reco_direction.right_ascension  for ev in events_ps])
ang_seps    = np.degrees([ev.true_direction.separation(ev.reco_direction)
                          for ev in events_ps])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
src_dec = np.degrees(src.location.declination)
src_ra  = np.degrees(src.location.right_ascension)

ax = axes[0]
ax.scatter(reco_ras, reco_decs, s=12, alpha=0.6, color='steelblue', label='Reco')
ax.scatter([src_ra], [src_dec], s=200, marker='*', color='gold',
           zorder=5, label='True position', edgecolors='k', linewidths=0.5)
ax.set_xlabel('Right ascension [deg]')
ax.set_ylabel('Declination [deg]')
ax.set_title('Reconstructed directions')
ax.legend(fontsize=9)

ax = axes[1]
bins = np.logspace(-1, 4, 30)
ax.hist(reco_e_tev, bins=bins, color='steelblue', edgecolor='white', lw=0.5)
ax.set_xscale('log')
ax.set_xlabel('Reconstructed energy [TeV]')
ax.set_ylabel('Events / bin')
ax.set_title('Reco energy spectrum (E^-2.7)')

ax = axes[2]
ax.hist(ang_seps, bins=25, color='steelblue', edgecolor='white', lw=0.5)
ax.set_xlabel('Angular separation (true to reco) [deg]')
ax.set_ylabel('Events / bin')
ax.set_title('PSF spread')

plt.tight_layout()
plt.show()

### 3.2  True vs reconstructed energy

For steeply falling spectra the energy resolution causes more events to scatter *upward* in energy than downward (Eddington bias).  We can see this directly by comparing E_true and E_reco for the same events.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(true_e_tev, reco_e_tev, s=10, alpha=0.5,
                c=np.log10(true_e_tev), cmap='viridis')
lim = [min(true_e_tev.min(), reco_e_tev.min()) * 0.5,
       max(true_e_tev.max(), reco_e_tev.max()) * 2.0]
ax.plot(lim, lim, 'k--', lw=1, label='E_reco = E_true')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('True energy [TeV]')
ax.set_ylabel('Reconstructed energy [TeV]')
ax.set_title('Energy smearing')
plt.colorbar(sc, ax=ax, label='log10(E_true / TeV)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 3.3  Atmospheric background

Atmospheric neutrinos — produced by cosmic-ray air showers — are the dominant background for astrophysical searches.  Because they arrive from all directions they are modelled as an `ExtendedSource`, but the sampling workflow is identical to any other source type.  SPORE ships several pre-computed flux tables in `resources/atmo_flux_models.h5`:

| Group | CR spectrum | Hadronic interaction model |
|---|---|---|
| `honda2006` | Honda 2006 | — |
| `mceq_h3a_sibyll23d` | H3A | Sibyll 2.3d |
| `mceq_h4a_sibyll23d` | H4A | Sibyll 2.3d |
| `mceq_gsf_sibyll23d` | GSF | Sibyll 2.3d |

In [ ]:
ATM_H5 = str(REPO_ROOT / 'resources' / 'atmo_flux_models.h5')
flux_model = "mceq_h4a_sibyll23d"

atmo_src = ExtendedSource.from_config(
    {"flux": {"location": f"{ATM_H5}:{flux_model}"}},
)

print(f'Model: {flux_model}')
print('Building atmospheric sampler (n_dec=100, n_ra=100, n_e=100)...')
atmo_sampler = SourceSampler(
    south_polar, atmo_src,
    n_time_samples=50, n_dec=100, n_ra=100, n_e=100,
)
n_exp_atmo = atmo_sampler.expected_events('track', one_year)
print(f'Expected track events in one year: {n_exp_atmo:.2e}')

In [ ]:
events_atmo = atmo_sampler.sample_events('track', deltat=ureg.Quantity(15.0, 'minute'), seed=4)
print(f'Sampled {len(events_atmo)} atmospheric track events in 15 minutes')

In [ ]:
atmo_decs   = np.degrees([ev.true_direction.declination     for ev in events_atmo])
atmo_log10e = [np.log10(ev.reco_energy.to('GeV').magnitude) for ev in events_atmo]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Atmospheric tracks ({flux_model}) -- IceCube south pole, 15 min')

ax = axes[0]
ax.hist(atmo_decs, bins=np.linspace(-90, 90, 37),
        color='darkorange', edgecolor='white', lw=0.5)
ax.axvline(0, color='k', ls='--', lw=0.8, label='Horizon (dec = 0)')
ax.set_xlabel('True declination [deg]')
ax.set_ylabel('Events / 5 deg bin')
ax.set_title('Declination distribution')
ax.legend(fontsize=9)

ax = axes[1]
ax.hist(atmo_log10e, bins=25, color='darkorange', edgecolor='white', lw=0.5)
ax.set_xlabel('log10(E_reco / GeV)')
ax.set_ylabel('Events / bin')
ax.set_title('Reconstructed energy spectrum')

plt.tight_layout()
plt.show()

### 3.4  Good run list

Real detector data contains gaps — calibration runs, maintenance, data-quality failures.  A *good run list* (GRL) encodes the valid livetime as a set of (start, stop) MJD intervals.  IceCube public data releases ship these as two-column CSV files under an `uptime/` directory.

`GoodRunList.from_path` accepts:

* a path to a **single uptime CSV**,
* a path to a **directory** — all `*_exp*.csv` files are loaded automatically,
* or a **list** of file and/or directory paths.

When a GRL is passed to `sample_events`, two things change relative to a plain `deltat`:

1. **Poisson mean** — computed from the *total livetime* (sum of run durations), not the calendar span.
2. **Event timestamps** — drawn within good runs, weighted by run duration, so gaps in the detector record produce gaps in the simulated event stream.

Below we build a synthetic GRL that mimics a realistic detector uptime (~88% live fraction) and compare the resulting event time series against uniform sampling over the same calendar span.

In [ ]:
# ------------------------------------------------------------------
# Build a synthetic GRL: 10 years of runs, each ~28 days, with
# ~4-day gaps between seasons (~88% live fraction overall).
# Written as a two-column MJD CSV matching the IceCube uptime format.
# ------------------------------------------------------------------
T0_MJD    = 55694.0   # approximate IC86-I start
RUN_DAYS  = 28.0
GAP_DAYS  =  4.0
N_RUNS    = 10 * 365 // int(RUN_DAYS + GAP_DAYS)  # ~113 runs over 10 yr

runs = []
t = T0_MJD
for _ in range(N_RUNS):
    runs.append((t, t + RUN_DAYS))
    t += RUN_DAYS + GAP_DAYS
runs = np.array(runs)

_grl_dir  = tempfile.mkdtemp()
_grl_file = os.path.join(_grl_dir, 'IC86_synthetic_exp.csv')
np.savetxt(_grl_file, runs, header='MJD_start[days]  MJD_stop[days]', comments='# ')

grl = GoodRunList.from_path(_grl_file)

calendar_span = ureg.Quantity(runs[-1, 1] - runs[0, 0], 'day')
print(f'Calendar span : {calendar_span:.1f}')
print(f'Total livetime: {grl.total_livetime:.1f}')
print(f'Live fraction : {grl.total_livetime / calendar_span:.1%}')

# ------------------------------------------------------------------
# Sample 10 years of point-source events using the GRL.
# The Poisson mean is based on the live time, not the calendar span.
# ------------------------------------------------------------------
events_grl     = sampler_ps.sample_events('track', grl=grl, seed=10)
events_uniform = sampler_ps.sample_events(
    'track', deltat=calendar_span,
    t=grl.t_start, seed=10,
)
print(f'\nGRL sampling   : {len(events_grl)} events  '
      f'(expected {sampler_ps.expected_events("track", grl=grl):.1f})')
print(f'Uniform deltat : {len(events_uniform)} events  '
      f'(expected {sampler_ps.expected_events("track", deltat=calendar_span):.1f})')

# ------------------------------------------------------------------
# Plot: good run windows (shaded) + event arrival times (ticks)
# ------------------------------------------------------------------
t0 = grl.t_start

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
fig.suptitle('Event arrival times: GRL-constrained vs uniform sampling (10 yr, TXS 0506+056)')

for ax, events, label, color in zip(
    axes,
    [events_grl, events_uniform],
    ['GRL-constrained', 'Uniform deltat'],
    ['steelblue', 'tomato'],
):
    # shade good run windows
    for start, stop in runs:
        ax.axvspan(start - t0, stop - t0, alpha=0.15, color='green', lw=0)
    # event ticks
    times = sorted(ev.time - t0 for ev in events)
    ax.eventplot(times, lineoffsets=0.5, linelengths=0.8, color=color, linewidths=0.6)
    ax.set_yticks([])
    ax.set_ylabel(label, fontsize=9)

axes[-1].set_xlabel('Elapsed time [days]')
axes[0].set_xlim(0, runs[-1, 1] - t0)
plt.tight_layout()
plt.show()

## 4  Extended source simulation

We model an isotropic diffuse E^-2 astrophysical flux.  The flux is written to a temporary HDF5 file and loaded via `ExtendedSource.from_config`.  The sampler runs in steady-state mode (diurnal-average A_eff).

In [ ]:
N_DEC, N_E  = 30, 40
sindecs     = np.linspace(-1.0, 1.0, N_DEC)
energies    = np.logspace(2.0, 6.0, N_E)   # 100 GeV -- 1 PeV

PHI_0 = 1e-18   # GeV^-1 cm^-2 s^-1 sr^-1 per species
phi   = PHI_0 * (energies / 1e5) ** (-2.0)

fluxes      = np.zeros((6, N_DEC, N_E))
fluxes[2]   = phi[np.newaxis, :]   # nu_mu
fluxes[3]   = phi[np.newaxis, :]   # nu_mu_bar

_tmpdir   = tempfile.mkdtemp()
FLUX_FILE = str(Path(_tmpdir) / 'isotropic_flux.h5')

with h5py.File(FLUX_FILE, 'w') as f:
    grp = f.create_group('flux')
    grp.create_dataset('sindecs',  data=sindecs)
    grp.create_dataset('energies', data=energies)
    grp.create_dataset('fluxes',   data=fluxes)

print('Flux file written to', FLUX_FILE)

In [ ]:
src_ext = ExtendedSource.from_config({'flux': {'location': f'{FLUX_FILE}:flux'}})

print('Building extended-source sampler (steady-state)...')
sampler_ext = SourceSampler(
    south_polar, src_ext,
    n_dec=30, n_ra=30, n_e=30,
    n_time_samples=100,
)

n_exp_ext = sampler_ext.expected_events('track', one_year)
print(f'Expected track events in one year: {n_exp_ext:.1f}')

In [ ]:
events_ext  = sampler_ext.sample_events('track', deltat=one_year, seed=2)
print(f'Sampled {len(events_ext)} events')

ext_decs   = np.degrees([ev.true_direction.declination     for ev in events_ext])
ext_log10e = [np.log10(ev.reco_energy.to('GeV').magnitude) for ev in events_ext]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Extended source (isotropic E^-2) -- IceCube, 1 year, steady-state')

ax = axes[0]
ax.hist(ext_decs, bins=np.linspace(-90, 90, 37),
        color='steelblue', edgecolor='white', lw=0.5)
ax.set_xlabel('True declination [deg]')
ax.set_ylabel('Events / 5 deg bin')
ax.set_title('Declination distribution\n(reflects sky-averaged A_eff)')

ax = axes[1]
ax.hist(ext_log10e, bins=25, color='steelblue', edgecolor='white', lw=0.5)
ax.set_xlabel('log10(E_reco / GeV)')
ax.set_ylabel('Events / bin')
ax.set_title('Reconstructed energy spectrum')

plt.tight_layout()
plt.show()

## 5  Sanity checks

### 5.1  Poisson statistics

Run 300 pseudo-experiments and check that the event count distribution matches Poisson(expected).

In [ ]:
N_PSEUDO = 3000
counts = np.array([
    len(sampler_ps.sample_events('track', deltat=one_year, seed=i))
    for i in range(N_PSEUDO)
])

from scipy.stats import poisson as scipy_poisson
k = np.arange(max(0, counts.min() - 3), counts.max() + 4)
pmf = scipy_poisson.pmf(k, n_exp)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(counts, bins=np.arange(counts.min() - 0.5, counts.max() + 1.5),
        density=True, color='steelblue', edgecolor='white', lw=0.5, label='Sampled')
ax.plot(k, pmf, 'o-', color='tomato', ms=4,
        label=f'Poisson(mu={n_exp:.2f})')
ax.axvline(counts.mean(), color='k', ls='--', lw=1,
           label=f'Sample mean = {counts.mean():.2f}')
ax.set_xlabel('Number of events')
ax.set_ylabel('Probability')
ax.set_title(f'Poisson sampling check ({N_PSEUDO} pseudo-experiments)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'Expected mean : {n_exp:.2f}')
print(f'Poisson std   : {n_exp**0.5:.2f}')
print(f'Sample mean   : {counts.mean():.2f}  (std {counts.std():.2f})')

### 5.2  Angular resolution vs reconstructed energy

The PSF should decrease with increasing energy. We sample a large fixed set and measure the median angular separation per energy bin.

In [ ]:
events_large = sampler_ps.sample_events('track', nevent=20000, seed=3)

e_reco = np.array([ev.reco_energy.to('TeV').magnitude for ev in events_large])
sep    = np.degrees([ev.true_direction.separation(ev.reco_direction)
                     for ev in events_large])

e_bins  = np.logspace(-1, 4, 12)
e_mids  = np.sqrt(e_bins[:-1] * e_bins[1:])
med_sep = []
for lo, hi in zip(e_bins[:-1], e_bins[1:]):
    mask = (e_reco >= lo) & (e_reco < hi)
    med_sep.append(np.median(sep[mask]) if mask.sum() > 0 else np.nan)
med_sep = np.array(med_sep)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(e_mids, med_sep, 'o-', color='steelblue', ms=5)
ax.set_xscale('log')
ax.set_xlabel('Reconstructed energy [TeV]')
ax.set_ylabel('Median angular separation [deg]')
ax.set_title('Angular resolution vs energy (2000 events, fixed count)')
plt.tight_layout()
plt.show()

## 6  Multi-detector joint simulation

`SourceSampler` accepts a list of `Detector` objects.  Every sampled event carries a `detector_id` field (0-indexed by position in the list) recording which instrument observed it.

Below we construct a second detector at a Mediterranean latitude (representative of KM3NeT geometry) and run the same TXS 0506+056 point source through both instruments simultaneously.  Both detectors load the IceCube PS-10yr IRF here; in a real analysis each would use its own response file.

In [ ]:
mediterranean = Detector.from_config({
    'properties': {'latitude': 36.3, 'longitude': 16.1, 'depth': 3500, 'medium': 'Water'},
    'response':   {'detector_response_file': RESPONSE_FILE},
})

In [ ]:
sampler_ps_both = SourceSampler([south_polar, mediterranean], src, n_time_samples=100)
joint_events = sampler_ps_both.sample_events("track", deltat=10 * one_year, seed=925)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
src_dec = np.degrees(src.location.declination)
src_ra  = np.degrees(src.location.right_ascension)

ax.scatter([src_ra], [src_dec], s=200, marker='*', color='gold',
           zorder=5, label='True position', edgecolors='k', linewidths=0.5)

ax.scatter(
    np.degrees([ev.reco_direction.right_ascension for ev in joint_events if ev.detector_id==0]),
    np.degrees([ev.reco_direction.declination for ev in joint_events if ev.detector_id==0]),
    s=40, 
    alpha=0.6,
    color='dodgerblue',
    label='South Polar',
    marker="x"
)

ax.scatter(
    np.degrees([ev.reco_direction.right_ascension for ev in joint_events if ev.detector_id==1]),
    np.degrees([ev.reco_direction.declination for ev in joint_events if ev.detector_id==1]),
    s=40, 
    alpha=0.6,
    color='crimson',
    label='Mediterranean',
    marker="+"
)

ax.set_xlabel('Right ascension [deg]')
ax.set_ylabel('Declination [deg]')
ax.set_title('Reconstructed directions')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
from collections import defaultdict

by_det = defaultdict(list)
for ev in joint_events:
    by_det[ev.detector_id].append(ev)

det_labels = {0: 'South Polar', 1: 'Mediterranean'}
det_colors = {0: 'dodgerblue', 1: 'crimson'}

t0 = min(ev.time for ev in joint_events)

fig, ax = plt.subplots(figsize=(8, 4))
for det_id, label in det_labels.items():
    det_events = by_det[det_id]
    if not det_events:
        continue
    days = sorted((ev.time - t0) for ev in det_events)
    ax.step(days, np.arange(1, len(days) + 1),
            where='post', color=det_colors[det_id], label=label, linewidth=1.5)

ax.set_xlabel('Elapsed time [days]')
ax.set_ylabel('Cumulative events')
ax.set_title('Event arrival time series (10 years, tracks)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 6  Saving and loading events

`write_events` and `read_events` from `spore.event_sampling.io` serialize event lists to HDF5.  Multiple groups (e.g., signal and background) can coexist in the same file; `list_groups` returns the available group names.

A Pandas/parquet round-trip is also available via `pd.DataFrame([e.to_dict() for e in events])` — see the `io` module docstring for details.

In [ ]:
from spore.event_sampling.io import write_events, read_events, list_groups

EVENTS_FILE = str(Path(tempfile.mkdtemp()) / 'run_events.h5')

# Write signal (point source) and background (atmospheric) into the same file under separate groups
write_events(events_ps,   EVENTS_FILE, group='sim1/signal')
write_events(events_atmo, EVENTS_FILE, group='sim1/background')

print('Groups in file:', list_groups(EVENTS_FILE))
print(f'  signal:     {len(events_ps)} events written')
print(f'  background: {len(events_atmo)} events written')

In [ ]:
signal_rt = read_events(EVENTS_FILE, group='sim1/signal')
bg_rt     = read_events(EVENTS_FILE, group='sim1/background')

print(f'Read back: {len(signal_rt)} signal, {len(bg_rt)} background events')

orig_e  = [ev.reco_energy.magnitude for ev in events_ps]
check_e = [ev.reco_energy.magnitude for ev in signal_rt]
assert np.allclose(orig_e, check_e)
print('Round-trip energy check passed.')

In [ ]:
ATM_H5 = str(REPO_ROOT / 'resources' / 'atmo_flux_models.h5')
flux_model = "mceq_h4a_sibyll23d"

atmo_src = ExtendedSource.from_config(
    {"flux": {"location": f"{ATM_H5}:{flux_model}"}},
)

print(f'Model: {flux_model}')
print('Building atmospheric sampler (n_dec=100, n_ra=100, n_e=100)...')
atmo_sampler = SourceSampler(
    south_polar, atmo_src,
    n_time_samples=50, n_dec=100, n_ra=100, n_e=100,
)
n_exp_atmo = atmo_sampler.expected_events('track', one_year)
print(f'Expected track events in one year: {n_exp_atmo:.2e}')

In [ ]:
sampler_atmo = SourceSampler(mediterranean, atmo_src, n_time_samples=100)
events = sampler_atmo.sample_events("track", deltat=10*one_year)

In [ ]:
decs = np.array([float(event.reco_direction.declination) for event in events])

In [ ]:
h, bins = np.histogram(np.sin(decs), bins=30)

In [ ]:
cents = (bins[1:] + bins[:-1]) / 2
plt.step(cents, h, where="mid")
plt.xlabel(r"$\sin(\delta)$")
plt.ylabel(r"$N_{\mathrm{event}}$")
plt.show()

In [ ]:
events = filter(lambda event: event.zenith > np.pi / 2, events)
decs = np.array([float(event.reco_direction.declination) for event in events])


In [ ]:
h, bins = np.histogram(np.sin(decs), bins=30)
cents = (bins[1:] + bins[:-1]) / 2
plt.step(cents, h, where="mid")
plt.xlabel(r"$\sin(\delta)$")
plt.ylabel(r"$N_{\mathrm{event}}$")
plt.xlim(-1,1)
plt.ylim(0,None)
plt.show()